# Objectif : Regression Logistique sur la proba de faire defaut (Y=1)

Premierement on va trouver pour chaque variable les modalités qui ont le tx de defaut le + élevé pour le mettre en valeur de reference

In [18]:
import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [19]:
df_discretise = pd.read_csv(filepath_or_buffer="../data/output/df_discretise.csv",
                 sep = ",")

In [20]:
def reorder_reference_by_risk(df, var, target="loan_status"):
    """
    Définit comme référence la modalité ayant le plus haut taux de défaut.
    """
    # Calcul du taux de défaut par modalité
    bad_rate = df.groupby(var)[target].mean().sort_values(ascending=False)
    ref = bad_rate.index[0]  # modalité à risque max
    print(f"→ Modalité de référence pour {var} : '{ref}' (taux de défaut = {bad_rate.iloc[0]:.3f})")

    # Reordonne les catégories
    df[var] = pd.Categorical(df[var], categories=bad_rate.index, ordered=True)
    return df, ref


In [22]:
categorical_vars = [
    "person_home_ownership",
    "loan_intent",
    "cb_person_default_on_file",
    "person_age_bin",
    "person_income_bin",
    "person_emp_length_bin",
    "loan_amnt_bin",
    "loan_percent_income_bin",
    "cb_person_cred_hist_length_bin"
]

df_encoded = df_discretise.copy()
ref_dict = {}

for var in categorical_vars:
    df_encoded, ref = reorder_reference_by_risk(df_encoded, var, target="loan_status")
    ref_dict[var] = ref

# Encodage final
X_encoded = pd.get_dummies(df_encoded[categorical_vars], drop_first=True)


→ Modalité de référence pour person_home_ownership : 'OTHER' (taux de défaut = 0.327)
→ Modalité de référence pour loan_intent : 'DEBTCONSOLIDATION' (taux de défaut = 0.290)
→ Modalité de référence pour cb_person_default_on_file : 'Y' (taux de défaut = 0.385)
→ Modalité de référence pour person_age_bin : 'person_age_Bin1' (taux de défaut = 0.259)
→ Modalité de référence pour person_income_bin : 'person_income_Bin1' (taux de défaut = 0.457)
→ Modalité de référence pour person_emp_length_bin : 'person_emp_length_Bin1' (taux de défaut = 0.287)
→ Modalité de référence pour loan_amnt_bin : 'loan_amnt_Bin5' (taux de défaut = 0.394)
→ Modalité de référence pour loan_percent_income_bin : 'loan_percent_income_Bin5' (taux de défaut = 0.711)
→ Modalité de référence pour cb_person_cred_hist_length_bin : 'cb_person_cred_hist_length_Bin1' (taux de défaut = 0.231)


In [24]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# --- 1️⃣ Définir les variables catégorielles ---
categorical_vars = [
    "person_home_ownership",
    "loan_intent",
    "cb_person_default_on_file",
    "person_age_bin",
    "person_income_bin",
    "person_emp_length_bin",
    "loan_amnt_bin",
    "loan_percent_income_bin",
    "cb_person_cred_hist_length_bin"
]

target = "loan_status"

# --- 2️⃣ Fonction pour réordonner les catégories selon le risque ---
def reorder_reference_by_risk(df, var, target="loan_status"):
    """
    Réordonne les modalités d'une variable catégorielle en mettant en premier
    celle ayant le plus fort taux de défaut (c'est elle qui deviendra la référence).
    """
    taux_defaut = df.groupby(var)[target].mean().sort_values(ascending=False)
    ref = taux_defaut.index[0]
    print(f"→ Modalité de référence pour {var} : '{ref}' (taux de défaut = {taux_defaut.iloc[0]:.3f})")

    df[var] = pd.Categorical(df[var], categories=taux_defaut.index, ordered=True)
    return df, ref

# --- 3️⃣ Application sur toutes les variables ---
df_encoded = df_discretise.copy()
ref_dict = {}

for var in categorical_vars:
    df_encoded, ref = reorder_reference_by_risk(df_encoded, var, target=target)
    ref_dict[var] = ref

# --- 4️⃣ Encodage en variables indicatrices (One-Hot Encoding) ---
X = pd.get_dummies(df_encoded[categorical_vars], drop_first=True)

# --- 5️⃣ Ajout de la constante ---
X = sm.add_constant(X)

# --- 6️⃣ Conversion en float ---
X = X.astype(float)

# --- 7️⃣ Cible ---
y = df_encoded[target]

# --- 8️⃣ Régression logistique ---
logit_model = sm.Logit(y, X)
result = logit_model.fit(disp=1)

# --- 9️⃣ Résumé du modèle ---
print(result.summary())

# --- 🔟 Récapitulatif des références utilisées ---
print("\n=== Modalités de référence ===")
for var, ref in ref_dict.items():
    print(f"{var} → {ref}")


→ Modalité de référence pour person_home_ownership : 'OTHER' (taux de défaut = 0.327)
→ Modalité de référence pour loan_intent : 'DEBTCONSOLIDATION' (taux de défaut = 0.290)
→ Modalité de référence pour cb_person_default_on_file : 'Y' (taux de défaut = 0.385)
→ Modalité de référence pour person_age_bin : 'person_age_Bin1' (taux de défaut = 0.259)
→ Modalité de référence pour person_income_bin : 'person_income_Bin1' (taux de défaut = 0.457)
→ Modalité de référence pour person_emp_length_bin : 'person_emp_length_Bin1' (taux de défaut = 0.287)
→ Modalité de référence pour loan_amnt_bin : 'loan_amnt_Bin5' (taux de défaut = 0.394)
→ Modalité de référence pour loan_percent_income_bin : 'loan_percent_income_Bin5' (taux de défaut = 0.711)
→ Modalité de référence pour cb_person_cred_hist_length_bin : 'cb_person_cred_hist_length_Bin1' (taux de défaut = 0.231)
Optimization terminated successfully.
         Current function value: 0.386934
         Iterations 7
                           Logit Reg

In [ ]:

y = df_discretise["loan_status"]


X_vars = [
    "person_home_ownership",
    "loan_intent",
    "cb_person_default_on_file",
    "person_age_bin",
    "person_income_bin",
    "person_emp_length_bin",
    "loan_amnt_bin",
    "loan_percent_income_bin",
    "cb_person_cred_hist_length_bin"
]

# --- 3️⃣ Encodage en variables indicatrices (One-Hot Encoding) ---
X = pd.get_dummies(df_discretise[X_vars], drop_first=True)

# --- 4️⃣ Ajout de la constante pour l’intercept ---
X = sm.add_constant(X)

# --- 5️⃣ Conversion explicite des booléens en valeurs numériques ---
X = X.astype(float)

# --- 6️⃣ Vérification ---
print(f"Dimensions de X : {X.shape}")
print(X.dtypes.value_counts())

# --- 7️⃣ Régression logistique ---
logit_model = sm.Logit(y, X)
result = logit_model.fit(disp=1)

# --- 8️⃣ Résumé du modèle ---
print(result.summary())

# --- 9️⃣ Calcul des Odds Ratios (effets multiplicatifs sur les odds de défaut) ---
odds_ratios = pd.DataFrame({
    "Variable": result.params.index,
    "Coefficient": result.params.values,
    "Odds_Ratio": np.exp(result.params.values),
    "p_value": result.pvalues
}).sort_values(by="Odds_Ratio", ascending=False)

print("\n=== Odds Ratios (effet multiplicatif sur les odds de défaut) ===")
print(odds_ratios)


,const,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE,cb_person_default_on_file_Y,...,loan_amnt_bin_loan_amnt_Bin4,loan_amnt_bin_loan_amnt_Bin5,loan_percent_income_bin_loan_percent_income_Bin2,loan_percent_income_bin_loan_percent_income_Bin3,loan_percent_income_bin_loan_percent_income_Bin4,loan_percent_income_bin_loan_percent_income_Bin5,cb_person_cred_hist_length_bin_cb_person_cred_hist_length_Bin2,cb_person_cred_hist_length_bin_cb_person_cred_hist_length_Bin3,cb_person_cred_hist_length_bin_cb_person_cred_hist_length_Bin4,cb_person_cred_hist_length_bin_cb_person_cred_hist_length_Bin5
0,1.0,False,True,False,True,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
1,1.0,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2,1.0,False,False,True,False,False,True,False,False,False,...,False,True,False,False,False,True,False,False,False,False
3,1.0,False,False,True,False,False,True,False,False,True,...,False,True,False,False,False,True,False,False,False,False
4,1.0,False,True,False,False,False,False,False,True,False,...,False,False,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29726,1.0,False,False,True,True,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,True
29727,1.0,False,False,False,False,False,False,True,False,False,...,False,False,True,False,False,False,False,False,False,True
29728,1.0,False,False,False,False,False,False,True,False,False,...,True,False,True,False,False,False,False,False,False,True
29729,1.0,False,False,True,False,True,False,False,False,False,...,False,True,False,False,False,True,False,False,False,True
